# Export pretrained baseline embeddings

The encoder runs in evaluation mode and the validation/test images are exported without augmentation. Rows remain paired with their stable `h5_index` values.


In [ ]:
# Configuration
from pathlib import Path
import sys

REPO_ROOT = next(
    root for root in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (root / "GLOSSARY.md").exists() and (root / "face_name_metric_learning").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
MANIFEST_PATH = REPO_ROOT / "data/processed/manifest.csv"
HDF5_PATH = REPO_ROOT / "data/processed/faces.h5"
ENVIRONMENT_RECORD_PATH = REPO_ROOT / "artifacts/face_name_metric_learning/setup_environment.json"
PRETRAINED_WEIGHTS_PATH = REPO_ROOT / "models/pretrained/iresnet100.pth"
PRETRAINED_MODULE = "face_name_metric_learning.iresnet"
OUTPUT_PATH = REPO_ROOT / "artifacts/face_name_metric_learning/embeddings/baseline_iresnet100.npz"
BATCH_SIZE = 128
EMBEDDING_DIMENSION = 512


In [ ]:
import json
from face_name_metric_learning import collect_environment_record, export_baseline_embeddings, load_embedding_package, validate_manifest_hdf5
from face_name_metric_learning.iresnet import load_iresnet100
assert MANIFEST_PATH.is_file(), f"Missing manifest: {MANIFEST_PATH}"
assert HDF5_PATH.is_file(), f"Missing prepared faces: {HDF5_PATH}"
assert ENVIRONMENT_RECORD_PATH.is_file(), f"Run setup notebook first: {ENVIRONMENT_RECORD_PATH}"
assert PRETRAINED_WEIGHTS_PATH.is_file(), f"Missing pretrained weights: {PRETRAINED_WEIGHTS_PATH}"
validate_manifest_hdf5(MANIFEST_PATH, HDF5_PATH)
with ENVIRONMENT_RECORD_PATH.open(encoding="utf-8") as stream: setup_record = json.load(stream)
assert setup_record["pretrained_weights"]["identifier"]
PRETRAINED_WEIGHT_SHA256 = setup_record["pretrained_weights"]["sha256"]
assert PRETRAINED_WEIGHT_SHA256


In [ ]:
device = collect_environment_record()["device"]
model = load_iresnet100(PRETRAINED_WEIGHTS_PATH, device, expected_sha256=PRETRAINED_WEIGHT_SHA256)
assert not model.training
output_path = export_baseline_embeddings(model, hdf5_path=HDF5_PATH, manifest_path=MANIFEST_PATH, output_path=OUTPUT_PATH, batch_size=BATCH_SIZE, device=device, expected_dimension=EMBEDDING_DIMENSION)
print(output_path)


In [ ]:
package = load_embedding_package(output_path, manifest_path=MANIFEST_PATH)
for split in ("validation", "test"):
    indices = package[f"{split}_h5_index"]
    embeddings = package[f"{split}_embeddings"]
    assert len(indices) == len(embeddings)
    assert embeddings.shape == (len(indices), EMBEDDING_DIMENSION)
    assert embeddings.dtype.name == "float32"
    print(split, embeddings.shape, embeddings.dtype, len(indices))
